# Reward Models & Moral Foundations Dictionary: Experimentation Pipeline

test

## Set-Up

Here, it should be easy to toggle: 

(1) if we're using KV-caching or the classic implementation.

(2) the dictionary we're using for our exhaustive search. (what we're cycling over as the assistant response that we'd like to score using our RM)

(3) the prompt phrasing variations we're using. (these are the templates for our user query -- 'I value [x] the most. What do you value most?', 'I think [x] is the greatest thing ever. What do you think is the greates thing ever?', etc.)

(4) the specific substitutions into the prompt phrasing (the [x] in 'I value [x] the most. What do you value most?')

### How to use the set-up block

The block below is the only place anything varies — it resolves the four toggles into a `runs` table (one row per output column) and a `dictionary` table (one row per candidate response), and the run cell just executes that plan. Nothing in the run cell needs editing to change the experiment.

**(1) Scoring path — `SCORING_MODE`.** `"kv_cache"` is the default and the one to use for real runs: it encodes the shared prompt prefix once and reuses its attention cache for every candidate, and it fixes the duplicate-BOS and decode/re-tokenize bugs documented in [`experiments/tokenization_bug_findings.md`](experiments/tokenization_bug_findings.md). `"fixed"` applies the same two fixes without caching (same cost as the original — isolates the fix from the speedup), and `"classic"` reproduces `get_reward_scores_from_response_token_ids` bugs and all, for comparison against the main-paper numbers. To compare paths, run each into its own `OUTPUT_DIR` — the checkpoint logic skips columns that already exist, so a second mode written to the same directory would be a no-op rather than a re-score.

**(2) Dictionary — `DICTIONARY_PATH`.** Defaults to `data/dictionaries/mfd2.0.dic`, the full-length MFD 2.0 (2,104 word-category pairs over 10 foundation/valence categories, 2,041 unique words after the 63 words filed under two foundations are deduplicated; the summary recommends the full-length version over the prototypicality-trimmed variants, and its validity is essentially the same). The loader also reads the other dictionaries already in that folder — `eloeverything_concepts.csv` (7,530 concepts) and `tokens_*.csv` (the full tokenizer vocabulary used by the main sweep) — so the same pipeline runs over any of the three. Use `DICTIONARY_CATEGORIES` to restrict to particular foundations and `MAX_ENTRIES` for a smoke test; note that unlike the main sweep the candidates here are **words, not single tokens**, so most are several tokens long and the two aren't directly comparable score-for-score.

**(3) Prompt phrasings — `config/mfd_prompts.yaml`.** Four templates, each pairing a value disclosure with the same question asked back, plus a matched baseline with the disclosure sentence deleted and nothing else changed. Select a subset with `TEMPLATES`. Adding a framing means adding a template *and* its baseline — the baseline is what makes a shift interpretable.

**(4) Substitutions — `config/mfd_substitutions.yaml`.** Four tiers, selected with `SUBSTITUTION_TIERS`. `primary` (10) and `extended` (21) are the MFD 2.0 seed words from Table 2 of the summary — the foundations' definitions, so they test whether the model shifts toward a foundation handed to it by name. Nine of Table 2's 40 seeds are absent: they're adjectives or verbs that don't fit the frame (`loyal`, `unnatural`), or nouns filed under a different foundation (`betrayal` is fairness.vice, not loyalty.vice). `sampled` (100) is 10 words per cell and is a **superset of both**: each cell's surviving seed words topped up with draws from that cell of the dictionary, so one selection gives the full 10-word set and the sub-groups stay distinguishable by their other tier label. The draws test whether the shift generalizes from any member of the category rather than its defining word. `control` (3) is non-moral. The default `["primary", "control"]` is the tractable full-coverage sweep; `sampled` is the expensive one (see sizing below). Tiers are a list per entry, so a seed word carries both `primary`/`extended` and `sampled`; a substitution is included if any of its tiers is selected.

The sampled words are drawn from `data/dictionaries/mfd2.0_typed.csv` — the dictionary with `word_type`, `frame_fit` and `number` labels per entry, built by `generate_typed_dictionary.py` — restricted to **singular** nouns and gerunds whose `frame_fit` is true. That's exactly the set that fits both frames: it fills a bare "I value ___ the most" without a determiner *and* agrees with "___ is the greatest thing ever". So `compassion` and `betraying` qualify; `hospital` (needs a determiner), `betray` (bare verb) and `rights` (plural, breaks the agreement) don't. loyalty.vice, MFD's smallest category, yields exactly 10 such words — which is what caps every cell at 10. Regenerate the sample with `python sample_mfd_substitutions.py --n 10 --seed 20260813`; it's seeded, so it reproduces exactly.

**Sizing a run.** Cost is `len(dictionary) x len(runs)` forward passes, and the set-up block prints both. The default is 2,041 entries x 56 columns = 114,296 sequences. Start with `MAX_ENTRIES = 50` and one template to confirm the plumbing, then clear both. `BATCH_SIZE` 384 suits a 40GB A100 — drop to 128 on a T4. KV-caching duplicates the cached prefix across the batch (`repeat_interleave`), so it doesn't necessarily buy batch-size headroom over `"fixed"` at equal length.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Set-up: resolves the four toggles into `dictionary` (candidate responses) and
# `runs` (output columns). Nothing below this cell needs editing to change the
# experiment. Runs on CPU — no model is loaded here.
# ─────────────────────────────────────────────────────────────────────────────
import re
import sys
from pathlib import Path

import pandas as pd
import yaml

# Repo root, whether this notebook is opened from the repo root locally or from
# /content/rm-optpessimal-personas on Colab.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "config").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

# ── (1) Scoring path ─────────────────────────────────────────────────────────
# "kv_cache" — cache the shared prompt prefix once per prompt; includes the
#              duplicate-BOS and decode/re-tokenize fixes. Use this for real runs.
# "fixed"    — same two fixes, no caching (full forward pass per candidate).
# "classic"  — reproduces get_reward_scores_from_response_token_ids exactly,
#              bugs included, for comparison against the main-paper numbers.
SCORING_MODE = "kv_cache"

MODEL_NAME = "Ray2333/GRM-Llama3.2-3B-rewardmodel-ft"  # must be in config/reward_models.yaml
BATCH_SIZE = 384  # 40GB A100; 128 on a T4

# One output directory per scoring mode, so the checkpoint-skip logic doesn't
# treat another mode's columns as already scored.
OUTPUT_DIR = REPO_ROOT / "data" / "mfd_reward_model_scores"
if SCORING_MODE != "kv_cache":
    OUTPUT_DIR = OUTPUT_DIR.with_name(f"{OUTPUT_DIR.name}_{SCORING_MODE}")

# ── (2) Dictionary: what gets scored as the assistant response ───────────────
DICTIONARY_PATH = REPO_ROOT / "data" / "dictionaries" / "mfd2.0.dic"
# Other dictionaries in that folder the loader also understands:
#   "mfd2.0_typed.csv"                                    (same 2,041 entries,
#       plus the word_type/frame_fit labels from generate_typed_dictionary.py;
#       those columns ride through into the scores CSV when you use this one)
#   "eloeverything_concepts.csv"                          (7,530 concepts)
#   "tokens_Ray2333--GRM-Llama3.2-3B-rewardmodel-ft.csv"  (full RM vocabulary)

DICTIONARY_CATEGORIES = None  # e.g. ["care.virtue", "care.vice"]; None = all
MAX_ENTRIES = None            # e.g. 50 for a smoke test; None = the whole dictionary

# How each entry is rendered as the assistant turn. "{entry}" is the bare word,
# which is what the one-word framings ask for. A sentence frame
# ("I value {entry} the most.") is a different experiment: it changes response
# length, and reward models are strongly length-sensitive, so don't mix frames
# within one output directory.
RESPONSE_TEMPLATE = "{entry}"

# ── (3)+(4) Prompt phrasings and their substitutions ─────────────────────────
PROMPTS_CONFIG = REPO_ROOT / "config" / "mfd_prompts.yaml"
SUBSTITUTIONS_CONFIG = REPO_ROOT / "config" / "mfd_substitutions.yaml"

TEMPLATES = None                              # e.g. ["value_most"]; None = all
# A substitution is included if it belongs to any tier listed here.
# "primary"  one MFD 2.0 seed word per foundation/valence (10)
# "extended" the other usable seed words for each cell (21 — nine of Table 2's
#            seeds are adjectives, verbs, or filed under another foundation)
# "sampled"  10 words per cell (100) — a superset of primary and extended:
#            each cell's seed words topped up with draws from that cell
# "control"  non-moral words (3)
# "sampled" multiplies the run: 100 substitutions x 4 templates is 404 columns,
# ~825k sequences. Pair it with a single template, or set MAX_ENTRIES, unless
# you have the GPU hours.
SUBSTITUTION_TIERS = ["primary", "control"]
SUBSTITUTIONS = None                          # e.g. ["fairness"]; None = all in the tiers
INCLUDE_BASELINES = True                      # the matched no-substitution prompts


def slugify(text):
    """Column-name slug — same convention as generate_persona_reward_model_scores.py."""
    return re.sub(r"[^a-z0-9]+", "_", text.strip().lower()).strip("_")


def load_dictionary(path):
    """Load a candidate-response list as a DataFrame [entry_id, text, categories].

    Understands the three dictionary formats in data/dictionaries/:
      *.dic  LIWC/MFD format — a `%`-delimited category header (id -> name),
             then one `word<TAB>id[<TAB>id...]` line per entry. 63 MFD 2.0 words
             appear under more than one category (e.g. `betray` is both
             fairness.vice and loyalty.vice), so entries are deduplicated by
             word and `categories` holds all of them, "|"-joined.
      *.csv  a concept/token list; the text column is `name`, `token_decoded`
             or `word`, whichever is present.
      *.txt  one entry per line.
    """
    path = Path(path)
    if path.suffix == ".dic":
        # Note: mfd2.0.dic is CR-terminated (classic Mac line endings); Python's
        # universal newlines handles that transparently.
        lines = [line.rstrip("\n") for line in path.read_text().splitlines()]
        marks = [i for i, line in enumerate(lines) if line.strip() == "%"]
        header, body = lines[marks[0] + 1:marks[1]], lines[marks[1] + 1:]
        cat_names = dict(line.split("\t", 1) for line in header if line.strip())

        entries = {}
        for line in body:
            if not line.strip():
                continue
            word, *cat_ids = line.split("\t")
            entries.setdefault(word, []).extend(cat_names[c] for c in cat_ids)
        df = pd.DataFrame({
            "text": list(entries),
            "categories": ["|".join(dict.fromkeys(c)) for c in entries.values()],
        })
    elif path.suffix == ".csv":
        raw = pd.read_csv(path)
        text_col = next(c for c in ("text", "name", "token_decoded", "word")
                        if c in raw.columns)
        df = pd.DataFrame({"text": raw[text_col].astype(str)})
        df["categories"] = raw["categories"] if "categories" in raw else ""
        # Labels from generate_typed_dictionary.py, kept if present.
        for extra in ("word_type", "frame_fit"):
            if extra in raw.columns:
                df[extra] = raw[extra]
        df = df.drop_duplicates("text")
    else:
        df = pd.DataFrame({"text": path.read_text().split("\n"), "categories": ""})
        df = df[df["text"].str.strip() != ""].drop_duplicates("text")

    df = df.reset_index(drop=True)
    df.insert(0, "entry_id", df.index)
    return df


# ── Resolve the dictionary ───────────────────────────────────────────────────
dictionary = load_dictionary(DICTIONARY_PATH)
if DICTIONARY_CATEGORIES:
    keep = set(DICTIONARY_CATEGORIES)
    mask = dictionary["categories"].apply(lambda c: bool(keep & set(c.split("|"))))
    dictionary = dictionary[mask].reset_index(drop=True)
if MAX_ENTRIES:
    dictionary = dictionary.head(MAX_ENTRIES).reset_index(drop=True)

dictionary["response"] = dictionary["text"].map(lambda t: RESPONSE_TEMPLATE.format(entry=t))

# Identity columns copied into the scores CSV ahead of the per-prompt columns.
ID_COLUMNS = [c for c in ("entry_id", "text", "categories", "word_type", "frame_fit")
              if c in dictionary.columns]

# ── Resolve the prompts ──────────────────────────────────────────────────────
prompts_cfg = yaml.safe_load(PROMPTS_CONFIG.read_text())
templates = prompts_cfg["templates"]
baselines = prompts_cfg.get("baselines", {})
if TEMPLATES:
    templates = {k: v for k, v in templates.items() if k in TEMPLATES}
    baselines = {k: v for k, v in baselines.items()
                 if k in {t["baseline_column"] for t in templates.values()}}

substitutions = yaml.safe_load(SUBSTITUTIONS_CONFIG.read_text())["substitutions"]
if SUBSTITUTION_TIERS:
    wanted = set(SUBSTITUTION_TIERS)
    substitutions = [s for s in substitutions if wanted & set(s["tiers"])]
if SUBSTITUTIONS:
    keep = set(SUBSTITUTIONS)
    substitutions = [s for s in substitutions if s["name"] in keep or slugify(s["name"]) in keep]

# One row per output column, carrying each prompt's substitution metadata.
# Saved alongside the scores so analysis never has to re-derive which prompt
# produced a column, or which word it primed with.
runs = []
if INCLUDE_BASELINES:
    for name, text in baselines.items():
        runs.append({"column": name, "prompt": text, "template": None,
                     "substitution": None, "foundation": None, "valence": None,
                     "tier": "baseline", "word_type": None, "baseline_column": None,
                     "group": None})
for template_name, template in templates.items():
    for sub in substitutions:
        runs.append({
            "column": f"{template_name}__{slugify(sub['name'])}",
            "prompt": template["text"].format(substitution=sub["name"]),
            "template": template_name,
            "substitution": sub["name"],
            "foundation": sub["foundation"],
            "valence": sub["valence"],
            "tier": "|".join(sub["tiers"]),
            "word_type": sub["word_type"],
            "baseline_column": template["baseline_column"],
            "group": template.get("group"),
        })
runs = pd.DataFrame(runs)

# ── Plan summary ─────────────────────────────────────────────────────────────
n_passes = len(dictionary) * len(runs)
print(f"model        {MODEL_NAME}")
print(f"scoring      {SCORING_MODE}  (batch size {BATCH_SIZE})")
print(f"dictionary   {DICTIONARY_PATH.name}: {len(dictionary)} entries"
      f"{f', categories {DICTIONARY_CATEGORIES}' if DICTIONARY_CATEGORIES else ''}")
print(f"prompts      {len(templates)} templates x {len(substitutions)} substitutions"
      f" + {len(baselines) if INCLUDE_BASELINES else 0} baselines = {len(runs)} columns")
print(f"cost         {len(dictionary)} x {len(runs)} = {n_passes:,} scored sequences")
print(f"output       {OUTPUT_DIR.relative_to(REPO_ROOT)}/{MODEL_NAME.replace('/', '--')}.csv")
print()
print("categories:", dict(dictionary["categories"].str.split("|").explode().value_counts()))
print()
print("first 3 prompts:")
for _, r in runs.head(3).iterrows():
    print(f"  {r['column']:38s} \"{r['prompt']}\"")
print(f"first 5 candidate responses: {dictionary['response'].head(5).tolist()}")

## Experiment Run Pipeline

Run an exhaustive token search as in colab_run_pipeline, but instead using the set-up described in the Set-Up section.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Run: score every dictionary entry as the assistant response, for every prompt
# in `runs`. Checkpoints after each column, so re-running resumes rather than
# restarting. Needs a GPU.
# ─────────────────────────────────────────────────────────────────────────────
from collections import defaultdict

import torch
from tqdm.auto import tqdm
from transformers.cache_utils import DynamicCache

from reward_model_support import RewardModel
from reward_model_registry import *  # registers each model's score extraction

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / f"{MODEL_NAME.replace('/', '--')}.csv"
runs_path = output_path.with_name(f"{output_path.stem}__runs.csv")

# ── Resume ───────────────────────────────────────────────────────────────────
if output_path.exists():
    scores_df = pd.read_csv(output_path)
    # Scores are positional, so a checkpoint written against a different
    # dictionary (or a different MAX_ENTRIES/RESPONSE_TEMPLATE) can't be
    # extended — its columns would silently misalign with these rows.
    if scores_df["text"].tolist() != dictionary["text"].tolist():
        raise ValueError(
            f"{output_path} was written for a different entry set "
            f"({len(scores_df)} rows vs {len(dictionary)} here). Point OUTPUT_DIR "
            "somewhere new, or restore the set-up toggles that produced it."
        )
else:
    scores_df = dictionary[ID_COLUMNS].copy()

pending = runs[~runs["column"].isin(scores_df.columns)]
print(f"{len(pending)}/{len(runs)} columns to score "
      f"({len(runs) - len(pending)} already in {output_path.name})")

# ── Scoring paths ────────────────────────────────────────────────────────────
def _cache_layers(cache):
    """Per-layer (keys, values) of a DynamicCache, across the transformers 4.56
    rename of .key_cache/.value_cache to .layers[i].keys/.values."""
    if hasattr(cache, "key_cache"):
        return list(zip(cache.key_cache, cache.value_cache))
    return [(layer.keys, layer.values) for layer in cache.layers]


def score_classic(rm, prompt, responses, batch_size):
    """The original path: each response goes in as chat-template *text* and the
    whole conversation is re-tokenized (double BOS, decode/re-tokenize seam —
    see experiments/tokenization_bug_findings.md). Kept bug-for-bug identical to
    RewardModel.get_reward_scores_from_response_token_ids so this notebook's
    numbers are comparable to the main sweep's.
    """
    scores = []
    for i in tqdm(range(0, len(responses), batch_size), leave=False):
        conversations = [
            [{"role": "user", "content": prompt}, {"role": "assistant", "content": r}]
            for r in responses[i:i + batch_size]
        ]
        formatted = [rm.tokenizer.apply_chat_template(c, tokenize=False)
                     for c in conversations]
        scores.extend(rm._calculate_batch_scores(formatted))
    return scores


def score_prefix_shared(rm, prompt, responses, batch_size, use_cache):
    """The "fixed" (use_cache=False) and "kv_cache" (True) paths, generalized
    from RewardModel.get_reward_scores_from_response_token_ids_{fixed,kv_cached}
    to multi-token responses — MFD entries are words, not single tokens, so a
    batch is only rectangular if its responses tokenize to the same length.
    Rather than pad (which would put the sequence-classification head's
    last-token lookup at the mercy of the padding side), responses are grouped
    by token length and each group is batched separately. Same number of forward
    passes, no padding anywhere.
    """
    device = rm.device
    prefix_ids, suffix_ids = rm._build_prefix_suffix_ids(prompt)
    prefix_ids, suffix_ids = prefix_ids.to(device), suffix_ids.to(device)
    prefix_len = prefix_ids.shape[1]

    if use_cache:
        with torch.no_grad():
            prefix_out = rm.model(
                input_ids=prefix_ids,
                attention_mask=torch.ones_like(prefix_ids),
                past_key_values=DynamicCache(),
                use_cache=True,
            )
        prefix_cache = prefix_out.past_key_values

    encoded = rm.tokenizer(list(responses), add_special_tokens=False)["input_ids"]
    by_length = defaultdict(list)
    for idx, ids in enumerate(encoded):
        by_length[len(ids)].append(idx)

    scores = [float("nan")] * len(responses)
    batches = [(length, group[i:i + batch_size])
               for length, group in sorted(by_length.items())
               for i in range(0, len(group), batch_size)]

    for length, group in tqdm(batches, leave=False):
        if length == 0:  # response tokenized to nothing (e.g. a blank entry)
            continue
        n = len(group)
        response_ids = torch.tensor([encoded[j] for j in group], device=device)

        if use_cache:
            new_ids = torch.cat([response_ids, suffix_ids.expand(n, -1)], dim=1)
            new_len = new_ids.shape[1]

            # The prefix was encoded once with batch dimension 1; broadcast it
            # across this batch. Built through update() rather than by assigning
            # cache.key_cache/.value_cache (as the persona sweep in
            # reward_model_support.py does) because those attributes don't exist
            # in transformers >= 4.56 — update() works on both sides of that
            # rename, and keeps the cache's own length bookkeeping correct.
            expanded_cache = DynamicCache()
            for layer_idx, (keys, values) in enumerate(_cache_layers(prefix_cache)):
                expanded_cache.update(keys.repeat_interleave(n, dim=0),
                                      values.repeat_interleave(n, dim=0),
                                      layer_idx)

            attention_mask = torch.ones(n, prefix_len + new_len, device=device)
            position_ids = torch.arange(prefix_len, prefix_len + new_len, device=device)
            position_ids = position_ids.unsqueeze(0).expand(n, -1)

            with torch.no_grad():
                outputs = rm.model(
                    input_ids=new_ids,
                    attention_mask=attention_mask,
                    past_key_values=expanded_cache,
                    position_ids=position_ids,
                    use_cache=False,
                )
        else:
            full_ids = torch.cat([
                prefix_ids.expand(n, -1),
                response_ids,
                suffix_ids.expand(n, -1),
            ], dim=1)
            with torch.no_grad():
                outputs = rm.model(input_ids=full_ids,
                                   attention_mask=torch.ones_like(full_ids))

        for idx, score in zip(group, rm._extract_scores_from_outputs(outputs)):
            scores[idx] = score
    return scores


# ── Sweep ────────────────────────────────────────────────────────────────────
if len(pending):
    reward_model = RewardModel.create(MODEL_NAME)
    if SCORING_MODE == "kv_cache" and reward_model.multi_gpu:
        raise ValueError(f"{MODEL_NAME} is multi_gpu: true — KV-cached scoring is "
                         'single-device only. Use SCORING_MODE = "fixed".')

    score_fns = {
        "classic": lambda p, r: score_classic(reward_model, p, r, BATCH_SIZE),
        "fixed": lambda p, r: score_prefix_shared(reward_model, p, r, BATCH_SIZE, False),
        "kv_cache": lambda p, r: score_prefix_shared(reward_model, p, r, BATCH_SIZE, True),
    }
    score_fn = score_fns[SCORING_MODE]
    responses = dictionary["response"].tolist()

    for _, run in tqdm(list(pending.iterrows()), desc="prompts"):
        print(f'{run["column"]}: "{run["prompt"]}"')
        scores_df[run["column"]] = score_fn(run["prompt"], responses)
        scores_df.to_csv(output_path, index=False, escapechar="\\")  # checkpoint

    del reward_model
    torch.cuda.empty_cache()

# Column -> prompt provenance, merged so resumed/partial runs keep earlier rows.
if runs_path.exists():
    previous = pd.read_csv(runs_path)
    runs_out = pd.concat([previous[~previous["column"].isin(runs["column"])], runs])
else:
    runs_out = runs
runs_out.to_csv(runs_path, index=False)

print(f"\nSaved {output_path} — {len(scores_df)} entries x "
      f"{len(scores_df.columns) - len(ID_COLUMNS)} prompt columns")
scores_df.head()

## Analysis/Figures

TODO: TBD